In [1]:
# IMPORTS
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

TIMEPOINTS = [0, 5, 10, 20, 40]   # matches the experimental data

In [2]:
def load_simulation_data(csv_path, n_timepoints=5):
    df = csv_path if isinstance(csv_path, pd.DataFrame) else pd.read_csv(csv_path)
    features = df.columns.tolist()
    n_samples = len(df) // n_timepoints
    data = df.values[:n_samples * n_timepoints].reshape(
        n_samples, n_timepoints, len(features)).astype(np.float32)
    print(f"Loaded {n_samples} samples, {n_timepoints} timepoints, {len(features)} features")
    return data, features


def get_metabolite_groups(features):
    groups = {}
    for i, col in enumerate(features):
        met, _, iso = col.rpartition("_")
        iso = iso.lstrip("M")
        if met and iso.isdigit():
            groups.setdefault(met, []).append((int(iso), i))
    return {m: [i for _, i in sorted(v)] for m, v in groups.items()}


def normalize_mid(data, groups):
    # Renormalize each metabolite block
    out = np.zeros_like(data)
    for idx in groups.values():
        block = data[..., idx]
        s = block.sum(-1, keepdims=True)
        out[..., idx] = block / np.where(s == 0, 1.0, s)
    return out

In [3]:
class MID_MDAE(nn.Module):
    def __init__(self, n_features, n_timepoints, groups,
                 hidden_dim=128, latent_dim=64, dropout=0.1):
        super().__init__()
        self.n_features, self.n_timepoints = n_features, n_timepoints
        self.groups, self.n_groups = groups, len(groups)
        in_dim, out_dim = 2 * n_features * n_timepoints, n_features * n_timepoints

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, latent_dim), nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
        group_ids = torch.zeros(n_features, dtype=torch.long)
        for gid, idx in enumerate(groups.values()):
            group_ids[idx] = gid
        self.register_buffer("group_ids", group_ids)

    def forward(self, x, mask):
        z = self.encoder(torch.cat([x.flatten(1), mask.flatten(1)], dim=1))
        logits = self.decoder(z).view(-1, self.n_timepoints, self.n_features)
        out = torch.zeros_like(logits)
        for idx in self.groups.values():                 # softmax within each metabolite
            out[..., idx] = torch.softmax(logits[..., idx], dim=-1)
        return out

In [4]:
def masked_mse(pred, target, mask):
    return ((pred - target) ** 2 * mask).sum() / (mask.sum() + 1e-10)


def train_mdae(model, loader, epochs=100, lr=1e-3, drop_low=0.30, drop_high=0.80):

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    T, Fdim, G = model.n_timepoints, model.n_features, model.n_groups
    print(f"\nTraining for {epochs} epochs...")
    for epoch in range(epochs):
        model.train(); total = n = 0
        for batch in loader:
            B = batch.size(0)
            p_drop = torch.empty(B, 1).uniform_(drop_low, drop_high)
            keep_met = (torch.rand(B, G) > p_drop).float()
            keep = keep_met[:, model.group_ids].unsqueeze(1).expand(B, T, Fdim)

            pred = model(batch * keep, keep)
            loss = masked_mse(pred, batch, 1 - keep)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); n += 1
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:>3d}/{epochs}  loss={total/n:.5f}")
    return model

## Validation with metabolite holdout

In [5]:
def validate_holdout(model, data, groups, holdout):
    N, T, Fdim = data.shape
    keep = np.ones(Fdim, np.float32)
    for met in holdout:
        if met in groups:
            keep[groups[met]] = 0.0

    model.eval()
    x = torch.tensor(data) * torch.tensor(keep)
    mask = torch.tensor(keep).view(1, 1, Fdim).expand(N, T, Fdim)
    with torch.no_grad():
        preds = model(x, mask).numpy()
    obs = keep.astype(bool)
    preds[:, :, obs] = data[:, :, obs]                    # keep observed values exact

    results = {}
    for met in holdout:
        if met not in groups:
            continue
        diff = preds[:, :, groups[met]] - data[:, :, groups[met]]
        results[met] = {"RMSE": float(np.sqrt((diff ** 2).mean())),
                        "MAE":  float(np.abs(diff).mean())}
    results["OVERALL"] = {"RMSE": float(np.mean([r["RMSE"] for r in results.values()])),
                          "MAE":  float(np.mean([r["MAE"]  for r in results.values()]))}
    return results


def print_validation_results(results):
    for met, m in results.items():
        print(f"  {met:<15s} RMSE={m['RMSE']:.5f}  MAE={m['MAE']:.5f}")

In [6]:
def perturb_mids(data, groups, noise=0.3, seed=42):
    # Multiplicative noise per isotopomer, then renormalize per metabolite.
    rng = np.random.default_rng(seed)
    out = data.copy()
    for idx in groups.values():
        block = out[:, :, idx] * rng.uniform(1 - noise, 1 + noise, out[:, :, idx].shape)
        block = np.clip(block, 0, None)
        s = block.sum(2, keepdims=True)
        out[:, :, idx] = block / np.where(s == 0, 1.0, s)
    return out

In [7]:
def load_mdv_table(tsv_path, sim_features, groups, timepoints=TIMEPOINTS):
    df = pd.read_csv(tsv_path, sep="\t")
    F = len(sim_features)
    tp_index = {t: i for i, t in enumerate(timepoints)}

    exp_array = np.zeros((len(timepoints), F), np.float32)
    obs_mask = np.zeros(F, bool)
    input_rows, seen = [], {}

    for _, row in df.iterrows():
        frag = row["#fragment_ID"]
        metab = frag.rsplit("_", 1)[0]
        t = int(row["time (s)"])
        input_rows.append({"fragment_ID": frag, "metabolite": metab, "time": t,
                           "mean_str": str(row["mean"]).strip(),
                           "sd_str": str(row["sd"]).strip()})
        seen[metab] = frag
        if metab not in groups or t not in tp_index:
            continue
        means = [float(x) for x in str(row["mean"]).split(",")]
        for k, col in enumerate(groups[metab]):
            if k < len(means):
                exp_array[tp_index[t], col] = means[k]
                obs_mask[col] = True

    print(f"\nMDV table loaded: {tsv_path}")
    print(f"  Fragments: {len(seen)}: {sorted(seen)}")
    print(f"  Observed isotopomer columns: {int(obs_mask.sum())}/{F}")
    return exp_array, obs_mask, input_rows


def write_mdv_table(out_path, mean_imp, std_imp, groups, input_rows,
                    timepoints=TIMEPOINTS, output_metabolites=None):

    obs_lookup = {r["metabolite"]: r["fragment_ID"] for r in input_rows}
    obs_strings = {(r["metabolite"], r["time"]): (r["mean_str"], r["sd_str"]) for r in input_rows}

    if output_metabolites is None:    
        ordered = [r["metabolite"] for r in input_rows if r["metabolite"] in groups]
        ordered += [m for m in groups if m not in ordered]
        output_metabolites = list(dict.fromkeys(ordered))

    tp_index = {t: i for i, t in enumerate(timepoints)}
    rows_out, n_obs, n_imp = [], 0, 0
    for metab in output_metabolites:
        if metab not in groups:
            continue
        cols = groups[metab]
        carbons = "".join(str(c) for c in range(1, len(cols)))
        frag_id = obs_lookup.get(metab, f"{metab}_{carbons}")
        for t in timepoints:
            ti = tp_index[t]
            if (metab, t) in obs_strings:
                mean_str, sd_str = obs_strings[(metab, t)]; n_obs += 1
                
            else:
                def fmt(vals):
                    return ",".join(f"{v:.5f}" for v in vals)
                
                mean_str = fmt(mean_imp[ti, cols]) 
                sd_str = fmt(std_imp[ti, cols])
                n_imp += 1
            rows_out.append({"#fragment_ID": frag_id, "time (s)": t,
                             "mean": mean_str, "sd": sd_str})

    out_df = pd.DataFrame(rows_out, columns=["#fragment_ID", "time (s)", "mean", "sd"])
    out_df.to_csv(out_path, sep="\t", index=False)
    print(f"\nWrote MDV table: {out_path}")
    return out_df

In [16]:
def impute_missing(model, exp_array, groups, n_passes=100, mc_dropout=True,
                   obs_mask=None, sd_floor=0.001):
    # exp_array 
    T, F = model.n_timepoints, model.n_features
    assert exp_array.shape == (T, F), f"expected {(T, F)}, got {exp_array.shape}"

    mask = np.zeros((T, F), np.float32)
    if obs_mask is not None:
        mask[:] = obs_mask.astype(np.float32)[None, :]
    aligned = normalize_mid(exp_array, groups)
    obs = mask.astype(bool)

    n_obs = int(mask.sum() / T)
    print(f"\nImputation: {T} timepoints, observed features {n_obs}/{F} ({n_obs/F:.1%})")

    model.eval()
    if mc_dropout:                                        # keep dropout on for MC sampling
        for m in model.modules():
            if isinstance(m, nn.Dropout):
                m.train()
    n_forward = n_passes if mc_dropout else 1

    x = torch.tensor(aligned).unsqueeze(0)
    mask_t = torch.tensor(mask).unsqueeze(0)
    with torch.no_grad():
        samples = np.array([model(x, mask_t).squeeze(0).numpy() for _ in range(n_forward)])

    mean, std = samples.mean(0), samples.std(0)
    mean[obs] = aligned[obs]
    std = np.maximum(std, sd_floor); std[obs] = 0.0
    return mean, std

## 10-fold cross-validation

In [17]:
def run_kfold_cv(data, groups, n_splits=10, epochs=100, lr=1e-3, batch_size=64,
                 drop_low=0.30, drop_high=0.80, hidden_dim=128, latent_dim=64,
                 dropout=0.10, eval_mask_fraction=0.30, seed=42):
    N, T, Fdim = data.shape
    rng = np.random.default_rng(seed)
    folds = np.array_split(rng.permutation(N), n_splits)
    fold_rmse, fold_mae = [], []

    for k in range(n_splits):
        val_idx = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(n_splits) if j != k])
        print(f"\n{'='*60}\n  Fold {k+1}/{n_splits}  (train={len(train_idx)}, val={len(val_idx)})\n{'='*60}")

        loader = DataLoader(torch.FloatTensor(data[train_idx]), batch_size=batch_size, shuffle=True)
        model = MID_MDAE(Fdim, T, groups, hidden_dim, latent_dim, dropout)
        train_mdae(model, loader, epochs=epochs, lr=lr, drop_low=drop_low, drop_high=drop_high)

        model.eval()
        val = data[val_idx]
        masked = rng.random(val.shape) < eval_mask_fraction
        for s in range(len(val)):                          # at least one masked element per sample
            if not masked[s].any():
                masked[s, 0, 0] = True
        keep = (~masked).astype(np.float32)
        with torch.no_grad():
            preds = model(torch.tensor(val * keep), torch.tensor(keep)).numpy()

        diff = preds[masked] - val[masked]
        fold_rmse.append(float(np.sqrt((diff ** 2).mean())))
        fold_mae.append(float(np.abs(diff).mean()))
        print(f"  Fold {k+1} -> RMSE={fold_rmse[-1]:.5f}  MAE={fold_mae[-1]:.5f}")
    return fold_rmse, fold_mae

## Main pipeline

In [18]:
print("DENOISING AUTOENCODER\n")

sim_data, sim_features = load_simulation_data("MID_8000_constrained.csv", n_timepoints=5)
metabolite_groups = get_metabolite_groups(sim_features)
print(f"Metabolite groups: {len(metabolite_groups)}")

n_train = int(0.9 * len(sim_data))
train_data, val_data = sim_data[:n_train], sim_data[n_train:]
print(f"Train: {len(train_data)}   Val: {len(val_data)}")

model = MID_MDAE(len(sim_features), 5, metabolite_groups, dropout=0.10)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

loader = DataLoader(torch.FloatTensor(train_data), batch_size=64, shuffle=True)
model = train_mdae(model, loader, epochs=100, lr=1e-3)

# Load experimental MDV table; metabolites
exp_array, obs_mask, mdv_rows = load_mdv_table("mid_original.tsv", sim_features, metabolite_groups)
observed = {r["metabolite"] for r in mdv_rows if r["metabolite"] in metabolite_groups}
holdout = [m for m in metabolite_groups if m not in observed]
print(f"Observed metabolites: {len(observed)}   Holdout: {len(holdout)}")

# Three validation scenarios 
results_s1 = validate_holdout(model, val_data, metabolite_groups, holdout)
val_perturbed = perturb_mids(val_data, metabolite_groups, noise=0.3, seed=42)
results_s2 = validate_holdout(model, val_perturbed, metabolite_groups, holdout)

rand_data, _ = load_simulation_data("MID_800_sim.csv", n_timepoints=5)
results_s3 = validate_holdout(model, rand_data, metabolite_groups, holdout)

plot_three_scenarios([results_s1, results_s2, results_s3],
                     ["Original MID set", "30% perturbed", "Random pools"])

# Impute experimental data
mean_imp, std_imp = impute_missing(model, exp_array, metabolite_groups,
                                   n_passes=120, mc_dropout=True, obs_mask=obs_mask)
write_mdv_table("mid_imputed_hehe.tsv",mean_imp, std_imp, metabolite_groups, mdv_rows)

DENOISING AUTOENCODER

Loaded 8000 samples, 5 timepoints, 265 features
Metabolite groups: 50
Train: 7200   Val: 800
Parameters: 560,109

Training for 100 epochs...
  Epoch  10/100  loss=0.02296
  Epoch  20/100  loss=0.02300
  Epoch  30/100  loss=0.02350
  Epoch  40/100  loss=0.02299
  Epoch  50/100  loss=0.02286
  Epoch  60/100  loss=0.02274
  Epoch  70/100  loss=0.02285
  Epoch  80/100  loss=0.02282
  Epoch  90/100  loss=0.02277
  Epoch 100/100  loss=0.02288

MDV table loaded: mid_original.tsv
  Fragments: 29  ->  ['adpglcH', 'akgC', 'akgM', 'aspLC', 'aspLM', 'dhapC', 'dhapH', 'f6pBH', 'fdpBC', 'fdpBH', 'g1pH', 'g6pAH', 'gluLC', 'gluLM', 'malLC', 'malLM', 'n2pglycH', 'n3pgC', 'n3pgH', 'pepC', 'pepH', 'pepM', 'r5pH', 'rb15bpH', 'ru5pDH', 's17bpH', 's7pH', 'succM', 'udpgC']
  Observed isotopomer columns: 163/265
Observed metabolites: 29   Holdout: 21
Loaded 800 samples, 5 timepoints, 265 features
Saved -> 'three_scenario_validation.png'

Imputation: 5 timepoints, observed features 163/2

,#fragment_ID,time (s),mean,sd
0,adpglcH_123456,0,"0.98993,0.00604,0.00403,0.00000,0.00000,0.0000...","0.08991,0.07015,0.06241,0.06134,0.00352,0.0179..."
1,adpglcH_123456,5,"0.98120,0.01160,0.00720,0.00000,0.00000,0.0000...","0.08991,0.07015,0.06241,0.06134,0.00352,0.0179..."
2,adpglcH_123456,10,"0.84165,0.09687,0.05191,0.00823,0.00135,0.0000...","0.08991,0.07015,0.06241,0.06134,0.00352,0.0179..."
3,adpglcH_123456,20,"0.52339,0.25898,0.17188,0.03633,0.00833,0.0010...","0.08991,0.07015,0.06241,0.06134,0.00352,0.0179..."
4,adpglcH_123456,40,"0.25932,0.33699,0.25288,0.11891,0.01288,0.0190...","0.08991,0.07015,0.06241,0.06134,0.00352,0.0179..."
...,...,...,...,...
245,xu5pDH_12345,0,"0.94748,0.05054,0.00085,0.00044,0.00033,0.00036","0.00100,0.00100,0.00100,0.00100,0.00100,0.00100"
246,xu5pDH_12345,5,"0.80664,0.17415,0.01765,0.00084,0.00037,0.00035","0.00948,0.00742,0.00186,0.00100,0.00100,0.00100"
247,xu5pDH_12345,10,"0.59571,0.31940,0.07522,0.00859,0.00053,0.00055","0.01469,0.00786,0.00561,0.00116,0.00100,0.00100"
248,xu5pDH_12345,20,"0.30972,0.41090,0.21499,0.05631,0.00740,0.00070","0.01250,0.00166,0.00802,0.00448,0.00100,0.00100"


In [ ]:
fold_rmse, fold_mae = run_kfold_cv(sim_data, metabolite_groups, n_splits=10,
                                   epochs=100, eval_mask_fraction=0.55, seed=42)